# T29 — Observability & Logging Lab (LangSmith & OpenTelemetry)

## Objective
Add structured logging and LLM call tracing to a production application. Instrument every LLM invocation with trace IDs, token usage metrics, latency breakdowns, and cost estimation logs.

### Observability Telemetry Architecture

```
User API Request
       │
       ▼
┌──────────────────────────────────────────┐
│ Instrumented FastAPI Application         │
└──────────────────┬───────────────────────┘
                   │
                   ▼
┌──────────────────────────────────────────┐
│  OpenTelemetry / LangSmith Trace Engine  │
└──────────────────┬───────────────────────┘
                   │
       ┌───────────┴───────────┐
       ▼                       ▼
┌──────────────┐        ┌──────────────┐
│  JSON Trace  │        │  Telemetry   │
│  Span Log    │        │  Dashboard   │
└──────────────┘        └──────────────┘
```



## 1. Environment Setup & Imports


In [1]:
import os
import time
import json
import uuid
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(dotenv_path=os.path.join("..", ".env"), override=True)
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in environment or .env file.")

client = OpenAI(api_key=api_key)
print("Environment initialized for Observability & Tracing Lab!")


Environment initialized for Observability & Tracing Lab!


## 2. Implement OpenTelemetry-Compatible Tracer


In [2]:
def trace_llm_call(prompt: str, model: str = "gpt-4o-mini"):
    trace_id = f"tr-{uuid.uuid4().hex[:8]}"
    t0 = time.time()
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a production assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    
    latency_ms = (time.time() - t0) * 1000
    answer = response.choices[0].message.content.strip()
    
    p_tokens = response.usage.prompt_tokens if hasattr(response, 'usage') and response.usage else len(prompt.split())
    c_tokens = response.usage.completion_tokens if hasattr(response, 'usage') and response.usage else len(answer.split())
    total_tok = response.usage.total_tokens if hasattr(response, 'usage') and response.usage else p_tokens + c_tokens
    
    # Cost calculation: $0.15/1M input, $0.60/1M output
    cost_usd = (p_tokens * 0.00000015) + (c_tokens * 0.0000006)
    
    trace_span = {
        "Trace ID": trace_id,
        "Model": model,
        "Prompt Tokens": p_tokens,
        "Completion Tokens": c_tokens,
        "Total Tokens": total_tok,
        "Latency (ms)": round(latency_ms, 2),
        "Cost ($)": round(cost_usd, 6),
        "Status": "SUCCESS"
    }
    
    return answer, trace_span

print("OpenTelemetry Tracer initialized!")


OpenTelemetry Tracer initialized!


## 3. Run Traced Workload & Inspect Telemetry Spans


In [3]:
test_prompts = [
    "Explain the concept of LLM observability in 2 sentences.",
    "What are the key metrics logged during RAG telemetry?",
    "Why is structured JSON logging preferred over plain text logs?"
]

traced_spans = []

for q in test_prompts:
    ans, span = trace_llm_call(q)
    traced_spans.append(span)

df_traces = pd.DataFrame(traced_spans)

print("="*80)
print("STRUCTURED LLM TRACE TELEMETRY LOG")
print("="*80)
print(df_traces.to_string(index=False))


STRUCTURED LLM TRACE TELEMETRY LOG
   Trace ID       Model  Prompt Tokens  Completion Tokens  Total Tokens  Latency (ms)  Cost ($)  Status
tr-9f2c148a gpt-4o-mini             30                 61            91       3572.36  0.000041 SUCCESS
tr-421cc6a5 gpt-4o-mini             28                361           389       5814.24  0.000221 SUCCESS
tr-d08c2e41 gpt-4o-mini             28                461           489       5667.13  0.000281 SUCCESS


## 4. Conclusion & Deliverable Summary

In **Task 29 (Observability & Logging)**:

1. **Instrumented App ([app.py](file:///C:/Users/tfd570/Desktop/month%202/t29/app.py))**: Implemented a FastAPI web application with structured JSON logging and OpenTelemetry trace span generation for every query.
2. **Logged Telemetry Fields**:
   - `trace_id`, `timestamp`, `user_id`, `prompt_tokens`, `completion_tokens`, `total_latency_ms`, and `estimated_cost_usd`.
3. **Observability Value**: Provides full audit trails, bottleneck identification, and cost accounting for production RAG deployments.

